<small><b>01 Data Cleaning</b> — Build <code>saas_cleaned.csv</code>: synthesize <code>searchable_text</code>, drop short copy, keep engineered features for ranking.</small>

In [1]:
from pathlib import Path
import sys

import pandas as pd

# Resolve project root whether the kernel cwd is repo/ or notebooks/
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.clean_data import (
    RAW_PATH,
    OUT_PATH,
    MIN_TEXT_CHARS,
    clean,
    missingness_report,
)

print("RAW:", RAW_PATH)
print("OUT:", OUT_PATH)
print("MIN_TEXT_CHARS:", MIN_TEXT_CHARS)

RAW: C:\Users\suxia\Desktop\Saas-Recommender(2026)\data\producthunt_features.csv
OUT: C:\Users\suxia\Desktop\Saas-Recommender(2026)\data\saas_cleaned.csv
MIN_TEXT_CHARS: 40


<small>1. Load CSV & check missing values — focus on <code>tagline</code> / <code>description</code>. Source has no free-text <code>topics</code>; we synthesize it from flags later.</small>

In [2]:
df = pd.read_csv(RAW_PATH)
print("shape:", df.shape)
print("n_columns:", len(df.columns))
display(df.head(3))

# "topics" is expected to be absent in the raw file (shown as N/A in the report)
focus = ["tagline", "description", "topics", "name"]
display(missingness_report(df, focus))
display(df[["tagline", "description"]].isna().sum())

shape: (5624, 31)
n_columns: 31


,id,name,tagline,description,votes_count,comments_count,maker_count,reviews_count,reviews_rating,media_count,...,topic_count,is_ai_product,is_saas_product,is_dev_tool,is_productivity,is_github_repo,has_custom_domain,daily_rank_clean,weekly_rank_clean,is_viral
0,1154630,Bluedot 2.1,Record on Apple Watch. Sync with Claude,Bluedot 2.1 brings your real-world conversatio...,247,21,5,12,4.67,4,...,3,0,0,0,1,0,1,1,8,1
1,1147823,Powabase,"Build AI apps with Postgres, RAG, and agents",Powabase is a backend-as-a-service for AI-nati...,223,31,4,0,0.00,6,...,3,1,0,1,0,0,1,2,11,1
2,1146179,Oasis Browser for Mac,A privacy-first AI browser you can train anony...,"Oasis is a refuge from noisy, scattered browsi...",173,49,62,2,5.00,9,...,3,1,0,0,1,0,1,3,16,1


,column,missing,missing_pct,note
0,tagline,0,0.0,ok
1,description,0,0.0,ok
2,topics,N/A,N/A,column absent in source
3,name,0,0.0,ok


tagline        0
description    0
dtype: int64

<small>2. Build <code>searchable_text</code> (embedding input) — <code>{name}. {tagline}. {description}. Topics: {topics}. Category: {main_category}</code>. Also derive <code>log_votes</code> / <code>engagement_ratio</code> and drop rows with tagline+description shorter than <code>MIN_TEXT_CHARS</code>.</small>

In [3]:
# clean() synthesizes topics/main_category/searchable_text, filters short text, adds ranking features
cleaned, stats = clean(df, min_text_chars=MIN_TEXT_CHARS)
stats

{'raw_rows': 5624,
 'cleaned_rows': 5622,
 'dropped_short_text': 2,
 'min_text_chars': 40,
 'main_category_counts': {'General': 1748,
  'AI': 1657,
  'SaaS': 813,
  'Productivity': 733,
  'Developer Tools': 671}}

In [4]:
# Spot-check key fields that feed retrieval + ranking
sample_cols = [
    "name", "tagline", "topics", "main_category",
    "votes_count", "log_votes", "engagement_ratio", "searchable_text",
]
display(cleaned[sample_cols].head(5))
print("\n--- searchable_text example (first 500 chars) ---\n")
print(cleaned.loc[0, "searchable_text"][:500])

,name,tagline,topics,main_category,votes_count,log_votes,engagement_ratio,searchable_text
0,Bluedot 2.1,Record on Apple Watch. Sync with Claude,Productivity,Productivity,247,5.513429,0.084677,Bluedot 2.1. Record on Apple Watch. Sync with ...
1,Powabase,"Build AI apps with Postgres, RAG, and agents","AI, Developer Tools",AI,223,5.411646,0.138393,"Powabase. Build AI apps with Postgres, RAG, an..."
2,Oasis Browser for Mac,A privacy-first AI browser you can train anony...,"AI, Productivity",AI,173,5.159055,0.281609,Oasis Browser for Mac. A privacy-first AI brow...
3,zero.xyz,"Give your AI agent access to ~8k tools, APIs a...","AI, Productivity",AI,160,5.081404,0.279503,zero.xyz. Give your AI agent access to ~8k too...
4,Coworker AI,More AI for less spend with context-aware mode...,"AI, SaaS, Productivity",SaaS,138,4.934474,0.258993,Coworker AI. More AI for less spend with conte...



--- searchable_text example (first 500 chars) ---

Bluedot 2.1. Record on Apple Watch. Sync with Claude. Bluedot 2.1 brings your real-world conversations into Claude. Record conversations directly from your Apple Watch, then sync them with Claude through MCP. Capture customer calls, hallway chats, interviews, coffee meetings, and in-person conversations, without a laptop or meeting bot. Bluedot turns every conversation into searchable, AI-ready context that Claude can summarize, search, and act on.. Topics: Productivity. Category: Productivity


<small>3. Sanity-check kept columns — category mix and viral rate help validate filters before export.</small>

In [5]:
print("columns kept:", list(cleaned.columns))
print("main_category distribution:")
display(cleaned["main_category"].value_counts())
# Useful later for popularity / "viral vs niche" ranking strategies
print("is_viral rate:", cleaned["is_viral"].mean().round(3))
print("engagement_ratio describe:")
display(cleaned["engagement_ratio"].describe())

columns kept: ['id', 'name', 'tagline', 'description', 'votes_count', 'comments_count', 'topics', 'main_category', 'platforms', 'searchable_text', 'log_votes', 'engagement_ratio', 'maker_count', 'reviews_count', 'reviews_rating', 'media_count', 'launch_hour_utc', 'launch_day_of_week', 'is_weekend', 'is_optimal_launch_hour', 'is_optimal_launch_day', 'name_len_chars', 'tagline_len_words', 'desc_len_words', 'has_emoji_in_tagline', 'is_question_tagline', 'topic_count', 'is_ai_product', 'is_saas_product', 'is_dev_tool', 'is_productivity', 'is_github_repo', 'has_custom_domain', 'daily_rank_clean', 'weekly_rank_clean', 'is_viral']
main_category distribution:


main_category
General            1748
AI                 1657
SaaS                813
Productivity        733
Developer Tools     671
Name: count, dtype: int64

is_viral rate: 0.178
engagement_ratio describe:


count    5622.000000
mean        0.362490
std         0.301426
min         0.000000
25%         0.200000
50%         0.333333
75%         0.500000
max         5.000000
Name: engagement_ratio, dtype: float64

<small>4. Export cleaned table for vector search (<code>02_vector_search.ipynb</code>) and the Gradio dashboard.</small>

In [6]:
# Idempotent write: re-run this notebook anytime to refresh saas_cleaned.csv
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
cleaned.to_csv(OUT_PATH, index=False)
print(f"Wrote {OUT_PATH} -> {cleaned.shape[0]} rows, {cleaned.shape[1]} cols")

Wrote C:\Users\suxia\Desktop\Saas-Recommender(2026)\data\saas_cleaned.csv -> 5622 rows, 36 cols
